In [1]:
import pandas as pd

from eval.translation_metrics import calc_bleurt_score, calc_comet_score, calc_dbleu_score, calc_ter_score
import os
os.chdir("..")

/Users/zp3077/Personal/Projects/translations_exploration/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using default BLEURT-Base checkpoint for sequence maximum length 128. You can use a bigger model for better results with e.g.: evaluate.load('bleurt', 'bleurt-large-512').


INFO:tensorflow:Reading checkpoint /Users/zp3077/.cache/huggingface/metrics/bleurt/default/downloads/extracted/d8b1bce334445f0def1d1a1c4851abc76d19672f541e0dcc15bf5eb22cfed174/bleurt-base-128.


Reading checkpoint /Users/zp3077/.cache/huggingface/metrics/bleurt/default/downloads/extracted/d8b1bce334445f0def1d1a1c4851abc76d19672f541e0dcc15bf5eb22cfed174/bleurt-base-128.


INFO:tensorflow:Config file found, reading.


Config file found, reading.


...

Creating Eager Mode predictor.


INFO:tensorflow:Loading model.


Loading model.
Fingerprint not found. Saved model loading will continue.
path_and_singleprint metric could not be logged. Saved model loading will continue.


INFO:tensorflow:BLEURT initialized.


BLEURT initialized.
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 69442.12it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/zp3077/Personal/Projects/translations_exploration/.venv/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [2]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


True

In [ ]:


# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/gemma-3-12b-it")
model = AutoModelForImageTextToText.from_pretrained("google/gemma-3-12b-it")


/Users/zp3077/Personal/Projects/translations_exploration/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards:  80%|████████  | 4/5 [00:43<00:11, 11.82s/it]

In [ ]:
# Example usage
inputs = processor(text="Translate this text to French",  return_tensors="pt")
outputs = model(**inputs)

# Process the outputs as needed
print(outputs)

In [ ]:
def translate(input_text):
    output = llm(
        f"""Translate the following German text to English. Only return the translated text, nothing else.\n"
        "German: {input_text}r\n"
        "English:""",
        max_tokens=50,  # Limit the response length
        stop=["\n"],  # Stops generation at the first newline to avoid extra text
        temperature=0  # Makes the output deterministic
    )
    return output['choices'][0]['text']


In [ ]:


df = pd.read_csv('data/new_sample.csv')

In [ ]:
df['en_transalted'] = df['de'].apply(translate)

## Apply all the transaltion metrics.

In [ ]:
def calc_metrics(org_col: str , machine_translated_col: str, gd_translated_col: str, df: pd.DataFrame):
    '''
    This function will calculate the metrics for the given columns
    :param org_col: the column the text in original language
    :param machine_translated_col: the column with the machine translated text
    :param gd_translated_col: the column with the human translated text
    :param df: the dataframe with the data.
    :return:
    '''
    df['dbleu'] = df.apply(lambda x: calc_dbleu_score(x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)
    df['bleurt'] = df.apply(lambda x: calc_bleurt_score(x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)
    df['comet'] = df.apply(lambda x: calc_comet_score(x[org_col].lower(), x[gd_translated_col].lower(), x[machine_translated_col].lower()), axis=1)

    df[['ter_num_edits','ter_score']]  = df.apply(lambda x: pd.Series(calc_ter_score(x[gd_translated_col].lower(), x[machine_translated_col].lower())), axis=1)
    df['dbleu'] = df['dbleu'].astype(float)
    return df


In [ ]:
df = calc_metrics('de', 'en_transalted', 'en', df)

In [ ]:
df.head()

In [ ]:
df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe()

In [ ]:
def summarize_df(df):

    output = llm(
        f"""You are an AI assistant. Your task is to summarize the given pandas DataFrame into the main points. Please provide a concise summary for this dataframe :{df}""",
        max_tokens=512,  # Limit the response length
        stop=["\n"],  # Stops generation at the first newline to avoid extra text
        temperature=0  # Makes the output deterministic
    )
    return output['choices'][0]['text']


In [25]:
df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe()

,dbleu,comet,bleurt,ter_num_edits
count,50.000000,50.000000,50.000000,18.000000
mean,14.038990,0.556420,-0.953566,10.111111
std,24.610616,0.216739,0.927806,9.361303
min,0.000000,0.335561,-2.312927,0.000000
25%,0.000000,0.397738,-1.641890,3.250000
50%,0.000000,0.424278,-1.246623,7.000000
75%,18.180648,0.812985,0.030089,17.500000
max,100.000000,0.973265,0.949728,31.000000


- dbleu: Shows a wide range of scores with high variability, indicating significant differences in performance.
- comet: Has a moderate average score with low to moderate variability, suggesting more consistent performance.
- bleurt: Displays negative average scores with moderate to high variability, indicating a mix of positive and negative performance.
- ter_num_edits: Shows moderate average scores with high variability, reflecting a wide range of edit distances.

In [24]:
summarize_df(df[[ 'dbleu', 'comet', 'bleurt', 'ter_num_edits']].describe())

Llama.generate: 375 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    6772.80 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =     128.73 ms /     1 runs   (  128.73 ms per token,     7.77 tokens per second)
llama_perf_context_print:       total time =     129.98 ms /     2 tokens


''